In [1]:
import networkx as nx
import os, glob, json
from pathlib import Path
import polars as pl
from tqdm.auto import tqdm

In [2]:
required_files = [
    "TX_Main_Pool_or_Add_{src_hash}_formatted.xlsx",
    "TX_Internes_Pool_or_Add_{src_hash}_formatted.xlsx",
    "TX_ERCFull_Pool_or_Add_{src_hash}_formatted.xlsx",
    "TX_ERC_Traced_Cleaned_{src_hash}.xlsx",
]

In [3]:
folder_hashes = {}
valid_hashes = {}


root_dir = "data"
for path in glob.glob(f"{root_dir}/**", recursive=True):
    path_parts = path.split("/")
    
    # ensure data/ token / file structure
    if len(path_parts) < 3: continue
    root, folder, file = path_parts
    
    # log new hashes
    if folder not in folder_hashes:
        folder_hashes[folder] = set()
        valid_hashes[folder] = set()
    
    basename = file.split(".")[0]
    hash = [_ for _ in basename.split("_") if "0x" in _]
    hash = hash[0] if hash else None # files without 0x return hash = []
    
    if not hash: continue
    
    folder_hashes[folder].add(hash)

In [4]:
for folder, hashes in folder_hashes.items():    
    folder_files = [Path(_).name for _ in glob.glob(f"{root_dir}/{folder}/*")]
    for hash in hashes:
        if all(
            required_file.format(src_hash=hash) in folder_files 
            for required_file in required_files):
            valid_hashes[folder].add(hash)

In [5]:
for folder in sorted(folder_hashes):
    print(f"""Diff: {len(folder_hashes[folder]) - len(valid_hashes[folder])}\tKept:{len(valid_hashes[folder])}   {folder=}""")

Diff: 0	Kept:1   folder='0xe96_____643bd 24-04-05 48j'
Diff: 1	Kept:0   folder='0xf869___ef6f 24-04-05 all'
Diff: 0	Kept:1   folder='AFY'
Diff: 1	Kept:0   folder='ALTN'
Diff: 1	Kept:0   folder='BALD 23-07-30 1j'
Diff: 1	Kept:0   folder='BALD Full'
Diff: 1	Kept:0   folder='BLC'
Diff: 1	Kept:1   folder='BRETT 24-03-20 3j'
Diff: 0	Kept:1   folder='BRETT 24-03-28 2j'
Diff: 0	Kept:1   folder='ECT'
Diff: 2	Kept:0   folder='G9'
Diff: 1	Kept:1   folder='ORCH'
Diff: 0	Kept:1   folder='PEPE UniV2 24-04-05 all'
Diff: 0	Kept:1   folder='TOSHI 24-03-24 3j'
Diff: 0	Kept:1   folder='TOSHI 24-04-28 3j'
Diff: 7	Kept:0   folder='all'


In [6]:

COL_ORDERS = ['hash', 'source', 'from', 'to', 'value', 'token', "sub_type"]

PATH = "./data/{folder}/"
total_files=0
for folder, hash_set in tqdm(valid_hashes.items()):
    for src_hash in hash_set:
        path = PATH.format(folder=folder)
        # try:
        main = (
            pl.read_excel(
                f"{path}/TX_Main_Pool_or_Add_{src_hash}_formatted.xlsx",

                read_options=dict(
                    columns = ['hash', 'from', 'to', 'value'],
                    dtypes={"value":str},
                )
            )
            .with_columns(
                pl.lit("ETH").alias("token"),
                pl.lit("main").alias("source"),
                pl.lit("main").alias("sub_type")
            )
        )

        internal = (
            pl.read_excel(
                f"{path}/TX_Internes_Pool_or_Add_{src_hash}_formatted.xlsx",
                read_options=dict(
                    columns = ['hash', 'from', 'to', 'value'] + ["callType"],
                    dtypes={"value":str},
                )
            )
            .with_columns(
                pl.lit("ETH").alias("token"),
                pl.lit("internal").alias("source"),
                pl.col("callType").alias("sub_type")
            )
            .filter(
                pl.col("value") != "0",
                pl.col("sub_type") == "call"
            )
        )


        # https://ethereum.stackexchange.com/questions/65740/whats-the-reason-erc-20-standard-allows-0-value-transfer
        erc = (
            pl.read_excel(
                f"{path}/TX_ERCFull_Pool_or_Add_{src_hash}_formatted.xlsx",
                read_options=dict(
                    columns = ['hash',  'from', 'to', 'tokenSymbol', 'value'] + ["tx_type"],
                    # ignore_errors=True,
                    dtypes={"value":str},
                )
            )
            .with_columns(
                pl.col("tokenSymbol").alias("token"),
                pl.col("tx_type").alias("sub_type"),
                pl.lit("erc").alias("source")
            )
            .filter(
                pl.col("value") != "0"
            )
        )

        erc_types = erc.select("hash", "from", "to", )

        erc_traced_replaces = {
            "Tx hash": "hash",
            "Tx type": "type",
            "final_value"    : "ETH_adj"
        }

        erc_traced = (
            pl.read_excel(
                f"{path}/TX_ERC_Traced_Cleaned_{src_hash}.xlsx",
                read_options=dict(
                    columns = ['Tx hash', 'Tx type', 'final_value'],
                    ignore_errors=True,
                )
            )
            .with_columns([
                pl.col(k).alias(v)
                for k, v in erc_traced_replaces.items()
            ])
            .drop(erc_traced_replaces.keys())
        )

        df = pl.concat([
            main.select(COL_ORDERS), 
            internal.select(COL_ORDERS), 
            erc.select(COL_ORDERS)
        ]).sort("hash", "source", "from", "to")

        df = (
            df
            .with_columns(
                pl.col("value").cast(float),
                *[
                    pl.col(c).str.to_lowercase().alias(c)
                    for c in ["from", "to"]
                ]
            )
            .with_columns(pl.col("token").alias("orig_token"))
            .with_columns(
                pl.col("token") # the algo tags WETH only, so assuming WETH == ETH
                .str.replace("WETH", "ETH")
                .str.replace("ETH", "WETH")
            )
        )

        df = df.join(erc_traced, on="hash")

        df.write_parquet(f"./data/all/{src_hash}.pqt")

        total_files += 1
        # except FileNotFoundError: # should be extraneous now that we process valid hashes first
            # print(folder, src_hash)
print(total_files)

  0%|          | 0/16 [00:00<?, ?it/s]

9


In [7]:
valid_hashes

{'BLC': set(),
 'TOSHI 24-04-28 3j': {'0x4b0Aaf3EBb163dd45F663b38b6d93f6093EBC2d3'},
 'G9': set(),
 'ALTN': set(),
 'BRETT 24-03-20 3j': {'0xBA3F945812a83471d709BCe9C3CA699A19FB46f7'},
 'BALD Full': set(),
 'TOSHI 24-03-24 3j': {'0x4b0Aaf3EBb163dd45F663b38b6d93f6093EBC2d3'},
 '0xf869___ef6f 24-04-05 all': set(),
 'BALD 23-07-30 1j': set(),
 'ECT': {'0x5b49001c8b756ab44b6ec3bcd1ea3f3a1c3cd758'},
 'BRETT 24-03-28 2j': {'0xBA3F945812a83471d709BCe9C3CA699A19FB46f7'},
 '0xe96_____643bd 24-04-05 48j': {'0xe96df8f5ef1a8790415068c798765b07d57643bd'},
 'AFY': {'0xf43e889444e95a0429c32b0b601dc16edf90fdbf'},
 'all': set(),
 'ORCH': {'0x61738277ff9b92e91b8ee02c0b73e8369b79ccfc'},
 'PEPE UniV2 24-04-05 all': {'0x76fc8dbf2022ee75612cb74ec80caf6b3ccffb23'}}

In [8]:
files = glob.glob(f"{root_dir}/all/0x*.pqt")
chunks = []
for file in files:
    _df = (
        pl.scan_parquet(file)
        .with_columns(
            pl.lit(
                Path(file).stem
            ).alias("src_file")
        )
        .with_columns(
            pl.col(c).cast(float) for c in ["value", "ETH_adj"]
        )
        .collect()
    )
    chunks.append(_df)

In [9]:
df = pl.concat(chunks)

In [10]:
final_df = (
    df
    .filter(
        pl.n_unique("src_file").over("hash") == 1
    )
)
final_df.write_parquet(f"{root_dir}/all_merged.pqt")

In [11]:
hashes_containing_WETH = (
    df
    .filter(
        pl.col("token") == "WETH"
    )
    .pivot(
        index="hash",
        values="token",
        columns="sub_type",
        aggregate_function="len"
    )
    .with_columns(
        pl.sum_horizontal(pl.all().exclude("hash")).alias("total_WETH")
    )
)

hashes_containing_WETH = (
    hashes_containing_WETH
    .join(
        df.group_by("hash").len(),
        on="hash"
    )
    .with_columns(
        (pl.col("total_WETH")/pl.col("len")).alias("prop")
    )
    .fill_null(0)
)

In [12]:
hashes_containing_WETH.write_parquet(f"{root_dir}/all_eth_stats.pqt")

# Meeting discussions

In [13]:
# by tx hash
df.group_by("hash").agg(pl.n_unique("src_file").alias("cnt")).sort("cnt")

hash,cnt
str,u32
"""0x3d594c9ddd83e0cbbc1298dfd041…",1
"""0x0c697858b2dd5e596703568d2179…",1
"""0x344f37c92da8d1be5982e71abd85…",1
"""0x83ae6daf46c43a5232abd27b30e0…",1
"""0xe794707dca557829c5c57bb61c4d…",1
…,…
"""0x26a590dfa85a44f3354412af065a…",2
"""0x666fb7367e58f063048d858e153e…",2
"""0xcac27790e1e32b63d2970d73c1ad…",2


In [14]:
dupes = (
    df
    .lazy()
    .select(
        "hash", "src_file"
    )
    .unique()
    .filter(
        pl.n_unique("src_file").over("hash") > 1
    )
    .group_by("src_file").len()
    .collect()
)
dupes

src_file,len
str,u32
"""0x61738277ff9b92e91b8ee02c0b73…",3
"""0x5b49001c8b756ab44b6ec3bcd1ea…",3
"""0xf43e889444e95a0429c32b0b601d…",6


In [15]:
# folders that have partial overlap
for folder, hashes in valid_hashes.items():
    if any(hash in dupes["src_file"] for hash in hashes):
        print(folder)

ECT
AFY
ORCH


In [16]:
# missing final values?
src_hash = '0x4b0Aaf3EBb163dd45F663b38b6d93f6093EBC2d3'

pl.read_excel(f"data/TOSHI 24-04-28 3j/TX_ERC_Traced_Cleaned_{src_hash}.xlsx").sort("Date")

"""
~ first 50 values/file not computed
\exists ETH -> still null ==> disagreement between algos
"""

,Date,Block number,Tx hash,Tx type,From,actions,Fee value,Fee token,Nb tx,token 1 name,token 1 value,token 2 name,token 2 value,token 3 name,token 3 value,final_value
i64,str,i64,str,str,str,str,str,str,i64,str,f64,str,f64,str,str,f64
0,"""2024-04-28 05:06:39""",13745726,"""0xfb60dfb86495f8c7865516b9cdda…","""remove liquitidy: multicall""","""0xd6b3ecad8c3525efa153902d4a80…","""{'TOSHI': 8135.294592331336, '…",null,null,6,"""TOSHI""",-8135.294592,"""ETH""",-0.001269,null,null,null
1,"""2024-04-28 05:07:49""",13745761,"""0x3ebba6086cc10dfff08e436c5fb1…","""swap""","""0xd6b3ecad8c3525efa153902d4a80…","""{'TOSHI': -9405.304114748074, …",null,null,6,"""TOSHI""",9405.304115,"""TYBG""",-21938.278046,null,null,0.001156
2,"""2024-04-28 05:11:39""",13745876,"""0xc0d7b67e6c437b7f01d28241168b…","""swap""","""0x855854bf8724ecfdc8e1d647008b…","""{'TOSHI': 1980741.7311648766, …",null,null,16,"""TOSHI""",-1.9807e6,"""ETH""",0.25,null,null,0.25
3,"""2024-04-28 05:16:21""",13746017,"""0xb041c0f34b16e68a47c16f0b584d…","""remove liquitidy: multicall""","""0xf6af31547b59022dd42aecd39981…","""{'TOSHI': 554.8251856991359, '…",null,null,6,"""TOSHI""",-554.825186,"""ETH""",-0.00002,null,null,null
4,"""2024-04-28 05:16:46""",13746030,"""0xa25777d745ad23240e3f75205d74…","""transfer""","""0xf6af31547b59022dd42aecd39981…","""{'ETH': 4.6124819690997e-05}""",null,null,4,"""ETH""",-0.000046,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1915,"""2024-05-01 02:13:49""",13870141,"""0xd1ded6eab3b020297af59aa7a29a…","""swap""","""0x7282f88d1394ef57c6daa8c5df45…","""{'TOSHI': -2562744.067697784, …",null,null,14,"""TOSHI""",2.5627e6,"""USDC""",-946.826467,null,null,0.255386
1916,"""2024-05-01 02:19:05""",13870299,"""0x4c00f7d0dc17f62c6139018cbae9…","""swap""","""0x3b92958e46ee22a3286c374e5371…","""{'TOSHI': -1190351.496655731, …",null,null,11,"""TOSHI""",1.1904e6,"""ETH""",-0.356524,null,null,0.118605
1917,"""2024-05-01 02:32:19""",13870696,"""0xf69cd85f083705ec327879eacadb…","""remove liquitidy: collect""","""0x9936e1b3ef3cec8a20e113c61871…","""{'ETH': 0.037494286779591955, …",null,null,2,"""TOSHI""",-603437.549979,"""ETH""",-0.037494,null,null,null
